<div style="background:linear-gradient(135deg,#0f0c29,#302b63);padding:20px 24px;border-radius:10px;border-left:5px solid #fb923c;font-family:Arial,sans-serif;">
  <h2 style="margin:0;color:#fb923c;">🖼️&nbsp;Image Processing App — 9+ Effects</h2>
  <p style="margin:8px 0 0;color:#bbb;font-size:14px;">OpenCV effects: Grayscale · Cartoon · Pencil Sketch · Blur · Edge Detection · Oil Painting · Sepia · Inverted · Emboss · tkinter GUI</p>
</div>

## Overview

Load any image, apply one of 9+ OpenCV visual effects, preview the result, and save.

| Effect | Method |
|--------|--------|
| Grayscale | `cv2.cvtColor` |
| Cartoon | `cv2.stylization` |
| Pencil Sketch | `cv2.pencilSketch` |
| Blur | `cv2.GaussianBlur` |
| Edge Detection | `cv2.Canny` |
| Oil Painting | `cv2.xphoto.oilPainting` |
| Sepia | custom matrix transform |
| Inverted | `cv2.bitwise_not` |
| Emboss | custom kernel `cv2.filter2D` |

```bash
pip install opencv-python opencv-contrib-python numpy pillow
```

> **Note:** Launches a tkinter desktop window. Section 3 shows a pure matplotlib demo you can run inside the notebook without a GUI.

## 1. Effect Engine

In [ ]:
import cv2
import numpy as np

EFFECTS = [
    "Grayscale", "Cartoon", "Pencil Sketch", "Blur",
    "Edge Detection", "Oil Painting", "Sepia", "Inverted Colors", "Emboss"
]

def apply_effect(img, name):
    """Apply a named OpenCV effect to a BGR image. Returns BGR image."""
    if name == "Grayscale":
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    elif name == "Cartoon":
        return cv2.stylization(img, sigma_s=200, sigma_r=0.3)
    elif name == "Pencil Sketch":
        gray, _ = cv2.pencilSketch(img, sigma_s=60, sigma_r=0.1)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    elif name == "Blur":
        return cv2.GaussianBlur(img, (21, 21), 0)
    elif name == "Edge Detection":
        edges = cv2.Canny(img, 80, 200)
        return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    elif name == "Oil Painting":
        return cv2.xphoto.oilPainting(img, 7, 1)
    elif name == "Sepia":
        kernel = np.array([[0.272, 0.534, 0.131],
                           [0.349, 0.686, 0.168],
                           [0.393, 0.769, 0.189]])
        return np.clip(cv2.transform(img, kernel), 0, 255).astype(np.uint8)
    elif name == "Inverted Colors":
        return cv2.bitwise_not(img)
    elif name == "Emboss":
        kernel = np.array([[-2,-1, 0],
                           [-1, 1, 1],
                           [ 0, 1, 2]])
        return cv2.filter2D(img, -1, kernel)
    return img

## 2. Pure Notebook Demo — Apply All Effects to a Sample Image

No GUI needed. Provide any local image path in `IMAGE_PATH`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

IMAGE_PATH = "sample.jpg"  # <-- change to your image path

try:
    img_bgr = cv2.imread(IMAGE_PATH)
    if img_bgr is None:
        raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

    fig, axes = plt.subplots(3, 3, figsize=(14, 12))
    axes = axes.flatten()

    for ax, effect in zip(axes, EFFECTS):
        result = apply_effect(img_bgr.copy(), effect)
        ax.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        ax.set_title(effect, fontsize=11, fontweight="bold")
        ax.axis("off")

    plt.suptitle("9 OpenCV Effects Comparison", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(f"Demo skipped: {e}")
    print("Set IMAGE_PATH to a valid .jpg or .png file to see all 9 effects.")

## 3. Full GUI Application (tkinter)

Upload Image → Select Effect → Apply → Save.

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk

image           = None
processed_image = None
tk_image        = None

def load_image():
    global image
    path = filedialog.askopenfilename(filetypes=[("Images","*.jpg *.jpeg *.png")])
    if not path: return
    image = cv2.imread(path)
    if image is None:
        messagebox.showerror("Error","Failed to load image."); return
    display_image(image)

def display_image(img):
    global tk_image
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    pil.thumbnail((420, 420))
    tk_image = ImageTk.PhotoImage(pil)
    label.config(image=tk_image)
    label.image = tk_image

def apply_selected():
    global processed_image
    if image is None:
        messagebox.showerror("Error","Upload an image first!"); return
    processed_image = apply_effect(image.copy(), effect_var.get())
    display_image(processed_image)

def save_image():
    if processed_image is None:
        messagebox.showerror("Error","No processed image to save!"); return
    path = filedialog.asksaveasfilename(
        defaultextension=".jpg",
        filetypes=[("JPEG","*.jpg"),("PNG","*.png")])
    if path:
        cv2.imwrite(path, processed_image)
        messagebox.showinfo("Saved", f"Saved to {path}")

root = tk.Tk()
root.title("Image Effect App")
root.configure(bg="#1e1e1e")

tk.Label(root, text="Select Effect:", bg="#1e1e1e", fg="white",
         font=("Arial",12)).pack(pady=(12,2))
effect_var = tk.StringVar(value=EFFECTS[0])
tk.OptionMenu(root, effect_var, *EFFECTS).pack()

label = tk.Label(root, bg="#1e1e1e")
label.pack(pady=8)

for txt, cmd in [("Upload Image", load_image),
                  ("Apply Effect", apply_selected),
                  ("Save Image",   save_image)]:
    tk.Button(root, text=txt, command=cmd, bg="#00cc99", fg="white",
              font=("Arial",11,"bold"), width=18).pack(pady=3)

tk.Button(root, text="Exit", command=root.quit,
          bg="#ff6666", fg="white", font=("Arial",11), width=18).pack(pady=(3,12))

root.mainloop()